# CS781 Capstone - Phase 1: Legal LM Pretraining
## Architecture 3: Encoder-Decoder Transformer (mixed-objective foundation model)

- **Model**: 10-layer bidirectional encoder + 4-layer causal decoder, d_model=640, 10 heads, full cross-attention (~86M params)
- **Blocks**: RoPE in self-attention (from A3 Q3), RMSNorm pre-norm, SwiGLU FFN, one shared embedding for encoder, decoder and output head
- **Objectives**: 90% of updates mixed (80% continuation / 20% T5-style span corruption, homogeneous batches), last 10% task-matched 512-word -> 128-word continuation
- **Tokenizer**: our A1 BPE (16,384 tokens, unchanged) + appended `<cont>`, `<denoise>` and 62 sentinel tokens
- **Data**: all 4 official sources, document-contained examples, 5% document-level validation split
- **Training**: one resumable run, ~18h of active training across sessions; cosine LR over a horizon measured in the opening updates
- Design rationale and references: the Phase 1 report

**How to run**
- Session 1: attach the tokenizer dataset, GPU T4, Internet on, `HF` secret, then *Save Version -> Save & Run All*.
- It builds and caches the data, trains until 1.5h before the 12h limit, and saves `arch3_latest.pt`.
- Session 2+: attach the previous version's output as an input, set `RUN_MODE = "resume"` in the config cell, and *Save & Run All* again; it reloads the cached data and resumes exactly.
- A checkpoint is written every 1000 updates (and at least hourly), at the session cutoff, and on any crash outside the optimizer step.
- Generation and `submission.zip` run only in the session where training completes.

## 1. HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import hashlib
import glob
import shutil
import numpy as np
from collections import Counter, defaultdict

NOTEBOOK_START = time.time()  # session clock: the platform limit counts from here

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Model (unchanged): 10-layer bidirectional encoder + 4-layer causal decoder, ~86M params
MODEL_DIM = 640
MODEL_ENC_LAYERS = 10
MODEL_DEC_LAYERS = 4
MODEL_HEADS = 10           # d_head = 64
MODEL_FF = 1712            # SwiGLU: 2/3 * 4 * 640 = 1706.7, rounded up to a multiple of 8
ENC_SEQ_LEN = 704          # encoder positions (mode token + up to 703 prompt tokens)
DEC_SEQ_LEN = 320          # decoder positions (BOS + target shifted right)
MAX_ENC_LEN = ENC_SEQ_LEN
MAX_DEC_LEN = DEC_SEQ_LEN
MODEL_DROPOUT = 0.0

# Token IDs. The BPE vocab (0..16383) is unchanged; special tokens are appended after it.
BASE_VOCAB = 16384
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3           # already reserved in the A1 tokenizer
CONT_ID = BASE_VOCAB                                   # <cont>: continuation mode (Phase 1 inference)
DENOISE_ID = BASE_VOCAB + 1                            # <denoise>: span-corruption mode
N_SENTINELS = 62                                       # <extra_id_0> ... <extra_id_61>
SENTINEL_IDS = list(range(BASE_VOCAB + 2, BASE_VOCAB + 2 + N_SENTINELS))
VOCAB_SIZE = BASE_VOCAB + 2 + N_SENTINELS              # 16448 = 257 * 64
IGNORE_INDEX = -100                                    # padded target positions carry no loss

# Objectives (engineering choices, not established optima)
P_DENOISE = 0.20           # share of optimizer updates that are span corruption during the mixed phase
FINAL_TASK_FRAC = 0.10     # last 10% of planned updates: task-matched continuation only
SPAN_NOISE_DENSITY = 0.15  # fraction of original tokens corrupted
SPAN_MEAN_LEN = 3.0        # mean corrupted span length
SPAN_SRC_LEN = 760         # original tokens per denoising example -> input ~685, target ~153 tokens

# Data
TRAIN_TOKENS = 600_000_000  # document-aligned block tokens to pre-tokenize; the sampler starts a new
                            # shuffled epoch if the run needs more (no automatic token-based stop)
VAL_TOKENS = 5_000_000
BLOCK = ENC_SEQ_LEN + DEC_SEQ_LEN   # 1024
VAL_DOC_PCT = 5                     # document-level split by hash of the dedup key
TASK_PROMPT_WORDS = 512
TASK_TARGET_WORDS = 128
TASK_POOL_SIZE = 120_000            # word-based 512->128 examples for the final segment
EVAL_SET_SIZE = 250                 # held-out 512->128 examples from validation documents only

# Training
BATCH_SIZE = 4
GRAD_ACCUM = 8             # 32 examples per optimizer update
PEAK_LR = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
TRAIN_BUDGET_HOURS = 18.0  # active training time across all sessions
MEASURE_FROM, MEASURE_TO = 50, 300   # updates used to measure throughput and fix the update horizon
HORIZON_SAFETY = 0.95
SESSION_LIMIT_HOURS = 12.0
SESSION_RESERVE_HOURS = 1.5          # stop training this long before the limit (final eval, 1000 generations, saves)
CKPT_EVERY_UPDATES = 1000            # checkpoint every 1000 successful optimizer updates
CKPT_EVERY_MIN = 60                  # ...and at least every 60 min in case updates are slower than expected
VAL_EVERY_UPDATES = 1000
MAX_VAL_BATCHES = 100
EVAL_MID_FRACS = [0.5]               # extra held-out BERTScore milestone(s), as a fraction of planned updates
EVAL_ENABLED = True                  # set False automatically if the BERTScore model cannot be loaded
LOG_EVERY = 50

# Generation (initial configuration, unchanged)
TARGET_WORDS = 128
MAX_GEN_TOKENS = MAX_DEC_LEN - 1     # BOS takes one decoder position
TEMPERATURE = 0.4
TOP_P = 0.92
TOP_K = 50
REPETITION_PENALTY = 1.15

# Tokenizer
_tok_matches = glob.glob("/kaggle/input/**/legal_bpe_16384.json", recursive=True)
TOKENIZER_PATH = _tok_matches[0] if _tok_matches else "/kaggle/input/legal-bpe-tokenizer/legal_bpe_16384.json"
print(f"Found tokenizer at: {TOKENIZER_PATH}")

# Run mode: "auto" resumes if a previous Arch 3 checkpoint is attached, otherwise starts fresh.
# Set "resume" for sessions 2+ so a forgotten input fails in seconds instead of silently restarting from scratch.
RUN_MODE = "auto"

# Output and resume files. Names are Arch-3-specific so an attached Arch 1/2 dataset is never picked up.
OUTPUT_DIR = "/kaggle/working/capstone_arch3"
INPUT_ROOT = "/kaggle/input"
LATEST_CKPT = "arch3_latest.pt"
PREV_CKPT = "arch3_latest_prev.pt"          # the checkpoint before latest, kept as a fallback
FOUNDATION_CKPT = "arch3_foundation_prefinal.pt"
BEST_PHASE1 = "arch3_best_phase1_model.pt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_input(name):
    m = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    return m[-1] if m else None

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, default=float)
    os.replace(tmp, path)

def session_elapsed():
    return time.time() - NOTEBOOK_START

def find_resume_dir():
    """Folder holding the most advanced Arch 3 checkpoint among this session's output and all attached inputs
    (decided by the update count in the run_log.json written next to every checkpoint)."""
    dirs = set()
    for name in (LATEST_CKPT, PREV_CKPT):
        if os.path.exists(os.path.join(OUTPUT_DIR, name)):
            dirs.add(OUTPUT_DIR)
        dirs.update(os.path.dirname(p) for p in glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    best, best_u = None, -1
    for d in sorted(dirs):
        try:
            with open(os.path.join(d, "run_log.json")) as f:
                u = int(json.load(f)["update"])
        except Exception:
            u = 0
        if u > best_u:
            best, best_u = d, u
    return best, best_u

RESUME_DIR, RESUME_UPDATE = find_resume_dir()
if RESUME_DIR:
    print(f"Previous Arch 3 run found: {RESUME_DIR} (update {RESUME_UPDATE:,}) -> this session RESUMES it")
elif RUN_MODE == "resume":
    raise RuntimeError("RUN_MODE='resume' but no arch3_latest.pt is attached: add the previous version's output as an input")
else:
    print("No previous Arch 3 checkpoint attached -> this session STARTS A NEW RUN (and builds the data, ~1-2h)")

print(f"Vocab: {BASE_VOCAB} BPE + 2 mode + {N_SENTINELS} sentinel = {VOCAB_SIZE}")
print("Configuration loaded.")


## 4. BPE Tokenizer (from A1, our own implementation)

In [ ]:
%%writefile bpe_tokenizer.py
"""
BPE Tokenizer  - CS781 Assignment 1 implementation, optimized for Phase 1.

Speedups over original A1 code (same algorithm):
  1. fit() uses word frequencies + incremental pair-count updates
  2. encode() uses priority-based merging + word-level cache
  3. Checkpointing: saves progress every N merges so training survives Kaggle timeouts

Research-backed defaults:
  - vocab_size=16384: optimal for 100M-scale models (Zheng et al. 2024,
    "Scaling Laws with Vocabulary"). At 100M scale, 16K minimizes training
    cost; quality spread across 8K-256K is <2% BPB.
  - Corpus sample: 5-10M words is sufficient for merge stabilization.
"""

import json
import os
import time
from collections import defaultdict


class BPETokenizer:

    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos.extend(range(256))
        self.eow_id = len(self.itos)  # 260
        self.itos.append("</w>")
        self.merges = []
        self.vocab_size = len(self.itos)
        self._merge_rank = {}
        self._encode_cache = {}
        self._token_bytes = None
        self._ends_word = None

    # ------------------------------------------------------------------ #
    #  Training (with checkpointing)                                      #
    # ------------------------------------------------------------------ #

    def fit(self, corpus, checkpoint_dir=None, checkpoint_every=1000):
        ckpt_path = None
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f"bpe_v{self.target_vocab_size}.ckpt.json")

        resumed_merge = 0
        word_data = None

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Found checkpoint at {ckpt_path}, resuming...")
            with open(ckpt_path, "r") as f:
                ckpt = json.load(f)
            self.merges = [((a, b), c) for (a, b), c in ckpt["merges"]]
            self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
            self.itos.extend(range(256))
            self.itos.append("</w>")
            for pair, new_id in self.merges:
                self.itos.append(pair)
            self.vocab_size = len(self.itos)
            resumed_merge = len(self.merges)
            word_data = []
            for tokens, freq in ckpt["words"]:
                word_data.append([tokens, freq])
            print(f"  Resumed after {resumed_merge} merges (vocab: {self.vocab_size})")

        if word_data is None:
            print("Counting word frequencies...")
            word_freq = {}
            for word in corpus.split():
                word_freq[word] = word_freq.get(word, 0) + 1
            print(f"  {len(word_freq):,} unique words")

            word_data = []
            for word, freq in word_freq.items():
                tokens = [b + 4 for b in word.encode("utf-8")]
                tokens.append(self.eow_id)
                word_data.append([tokens, freq])

        print("Building pair index...")
        pair_counts = defaultdict(int)
        pair_words = defaultdict(set)

        for idx, (tokens, freq) in enumerate(word_data):
            seen = defaultdict(int)
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                seen[pair] += 1
            for pair, cnt in seen.items():
                pair_counts[pair] += cnt * freq
                pair_words[pair].add(idx)

        num_merges_total = self.target_vocab_size - 261
        num_merges_left = num_merges_total - resumed_merge
        print(f"Learning {num_merges_left} merges (of {num_merges_total} total)...")
        t0 = time.time()

        for merge_i in range(num_merges_left):
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] < 2:
                break

            new_id = len(self.itos)
            bp_a, bp_b = best_pair

            affected = list(pair_words.get(best_pair, set()))

            for idx in affected:
                tokens, freq = word_data[idx]

                old_pairs = defaultdict(int)
                for i in range(len(tokens) - 1):
                    old_pairs[(tokens[i], tokens[i + 1])] += 1
                for p, cnt in old_pairs.items():
                    pair_counts[p] -= cnt * freq
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_words[p].discard(idx)
                    if not pair_words[p]:
                        pair_words.pop(p, None)

                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == bp_a and tokens[i + 1] == bp_b:
                        new_tokens.append(new_id)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                word_data[idx][0] = new_tokens

                new_pairs = defaultdict(int)
                for i in range(len(new_tokens) - 1):
                    new_pairs[(new_tokens[i], new_tokens[i + 1])] += 1
                for p, cnt in new_pairs.items():
                    pair_counts[p] = pair_counts.get(p, 0) + cnt * freq
                    pair_words[p].add(idx)

            self.merges.append((best_pair, new_id))
            self.itos.append(best_pair)
            self.vocab_size = len(self.itos)

            done = resumed_merge + merge_i + 1
            if done % 1000 == 0:
                elapsed = time.time() - t0
                rate = (merge_i + 1) / elapsed
                eta = (num_merges_left - merge_i - 1) / rate if rate > 0 else 0
                print(f"  {done}/{num_merges_total} merges | "
                      f"{elapsed:.0f}s elapsed | "
                      f"~{eta / 60:.1f} min remaining")

            if ckpt_path and (merge_i + 1) % checkpoint_every == 0:
                self._save_checkpoint(ckpt_path, word_data)

        if ckpt_path:
            self._save_checkpoint(ckpt_path, word_data)

        self._rebuild_caches()
        total_time = time.time() - t0
        print(f"Done. Final vocab: {self.vocab_size} | Time: {total_time / 60:.1f} min")
        return self

    def _save_checkpoint(self, path, word_data):
        ckpt = {
            "merges": [[[a, b], c] for (a, b), c in self.merges],
            "words": [[tokens, freq] for tokens, freq in word_data],
        }
        with open(path, "w") as f:
            json.dump(ckpt, f)

    # ------------------------------------------------------------------ #
    #  Encoding                                                           #
    # ------------------------------------------------------------------ #

    def _rebuild_caches(self):
        self._merge_rank = {}
        for rank, ((a, b), new_id) in enumerate(self.merges):
            self._merge_rank[(a, b)] = (rank, new_id)
        self._encode_cache = {}

        self._token_bytes = {x + 4: bytes([x]) for x in range(256)}
        self._token_bytes[self.eow_id] = b""
        self._ends_word = {i + 4: False for i in range(256)}
        self._ends_word[self.eow_id] = True
        for (a, b), c in self.merges:
            self._token_bytes[c] = self._token_bytes[a] + self._token_bytes[b]
            self._ends_word[c] = self._ends_word[b]

    def _encode_word(self, word):
        tokens = [b + 4 for b in word.encode("utf-8")]
        tokens.append(self.eow_id)

        while len(tokens) >= 2:
            best_rank = float("inf")
            best_idx = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                r = self._merge_rank.get(pair)
                if r is not None and r[0] < best_rank:
                    best_rank = r[0]
                    best_idx = i

            if best_idx == -1:
                break

            _, new_id = self._merge_rank[(tokens[best_idx], tokens[best_idx + 1])]
            tokens = tokens[:best_idx] + [new_id] + tokens[best_idx + 2:]

        return tokens

    def encode(self, text):
        if not self._merge_rank and self.merges:
            self._rebuild_caches()

        result = []
        for word in text.split():
            cached = self._encode_cache.get(word)
            if cached is not None:
                result.extend(cached)
            else:
                tokens = self._encode_word(word)
                self._encode_cache[word] = tokens
                result.extend(tokens)
        return result

    # ------------------------------------------------------------------ #
    #  Decoding                                                           #
    # ------------------------------------------------------------------ #

    def decode(self, tokens):
        if self._token_bytes is None:
            self._rebuild_caches()

        parts = []
        for i, token in enumerate(tokens):
            b = self._token_bytes.get(token)
            if b is not None:
                parts.append(b)
                if self._ends_word.get(token, False) and i < len(tokens) - 1:
                    parts.append(b" ")

        return b"".join(parts).decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ #
    #  Save / Load                                                        #
    # ------------------------------------------------------------------ #

    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "target_vocab_size": self.target_vocab_size,
            "merges": [[[a, b], c] for (a, b), c in self.merges],
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Tokenizer saved to {path} ({self.vocab_size} tokens)")

    @classmethod
    def load(cls, path):
        with open(path, "r") as f:
            data = json.load(f)
        tok = cls(data["target_vocab_size"])
        tok.merges = [((a, b), c) for (a, b), c in data["merges"]]
        for pair, new_id in tok.merges:
            tok.itos.append(pair)
        tok.vocab_size = len(tok.itos)
        tok._rebuild_caches()
        return tok

    @property
    def pad_token_id(self):
        return 0

    @property
    def unk_token_id(self):
        return 1

    @property
    def bos_token_id(self):
        return 2

    @property
    def eos_token_id(self):
        return 3

    def __len__(self):
        return self.vocab_size

## 5. Load Tokenizer and Append Special Tokens

The 16,384 BPE IDs are unchanged; `<cont>`, `<denoise>` and `<extra_id_0..61>` are appended after them (vocab 16,448).

In [ ]:
from bpe_tokenizer import BPETokenizer

tokenizer = BPETokenizer.load(TOKENIZER_PATH)
assert tokenizer.vocab_size == BASE_VOCAB, f"expected {BASE_VOCAB}-token BPE, got {tokenizer.vocab_size}"
print(f"Loaded tokenizer: vocab_size={tokenizer.vocab_size}")

# Special tokens are appended after the BPE vocab, so every existing BPE ID keeps its meaning.
SPECIAL_TOKENS = {"<pad>": PAD_ID, "<unk>": UNK_ID, "<s>": BOS_ID, "</s>": EOS_ID,
                  "<cont>": CONT_ID, "<denoise>": DENOISE_ID}
SPECIAL_TOKENS.update({f"<extra_id_{i}>": t for i, t in enumerate(SENTINEL_IDS)})
NEVER_GENERATE = sorted({PAD_ID, UNK_ID, BOS_ID, EOS_ID, CONT_ID, DENOISE_ID, *SENTINEL_IDS})

shutil.copy(TOKENIZER_PATH, os.path.join(OUTPUT_DIR, "legal_bpe_16384.json"))
with open(os.path.join(OUTPUT_DIR, "special_tokens.json"), "w") as f:
    json.dump({"base_vocab": BASE_VOCAB, "vocab_size": VOCAB_SIZE, "special_tokens": SPECIAL_TOKENS,
               "tokenizer_file": "legal_bpe_16384.json"}, f, indent=1)

# sanity check
test_text = "The Supreme Court of India hereby declares that s. 6(1)(a) of the Act applies."
enc = tokenizer.encode(test_text)
assert max(enc) < BASE_VOCAB
print(f"Encode/decode: {tokenizer.decode(enc)!r}")
print(f"Specials decode to nothing: {tokenizer.decode([CONT_ID, SENTINEL_IDS[0], EOS_ID] + enc[:3])!r}")
print(f"Tokens per word: {len(enc)/len(test_text.split()):.2f}")


## 6. Data: Build or Reload, with Audit

- Session 1 streams the corpus, dedups globally (MD5 of the first 500 chars, across all four sources), assigns each document to validation by the hash of that key (5%), and builds: document-aligned 1024-token blocks (random offset), a pool of word-based 512 -> 128 task examples, and a fixed held-out eval set from validation documents.
- Everything is cached to the output folder; later sessions reload it, so the split, blocks and pools are identical across sessions.

In [ ]:
try:
    import ijson
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'ijson', '-q'], check=True)
    import ijson

import array
import gc
from huggingface_hub import hf_hub_download

CACHE = {
    "train": "arch3_train_blocks.npy",
    "val": "arch3_val_blocks.npy",
    "task": "arch3_task_pool.npz",
    "eval": "arch3_eval_set.json",
    "meta": "arch3_data_meta.json",
}
def find_cached(name):
    if RESUME_DIR and os.path.exists(os.path.join(RESUME_DIR, name)):
        return os.path.join(RESUME_DIR, name)
    local = os.path.join(OUTPUT_DIR, name)
    return local if os.path.exists(local) else find_input(name)

cached = {k: find_cached(v) for k, v in CACHE.items()}
if RESUME_DIR and not all(cached.values()):
    raise RuntimeError(f"resuming from {RESUME_DIR} but its data cache is incomplete: {cached}")

if all(cached.values()):
    # Later sessions: reuse the exact data built in session 1 (same split, same blocks, same pools).
    print("Loading data cache from a previous session:")
    for k, p in cached.items():
        print(f"  {k}: {p}")
        dst = os.path.join(OUTPUT_DIR, CACHE[k])
        if os.path.abspath(p) != os.path.abspath(dst):
            shutil.copy(p, dst)  # carry the cache forward so the next session's input has it too
    train_blocks = np.load(os.path.join(OUTPUT_DIR, CACHE["train"]), mmap_mode="r")
    val_blocks = np.load(os.path.join(OUTPUT_DIR, CACHE["val"]), mmap_mode="r")
    _t = np.load(os.path.join(OUTPUT_DIR, CACHE["task"]))
    task_tokens, task_offsets, task_plen = _t["tokens"], _t["offsets"], _t["plen"]
    with open(os.path.join(OUTPUT_DIR, CACHE["eval"])) as f:
        eval_set = json.load(f)
    with open(os.path.join(OUTPUT_DIR, CACHE["meta"])) as f:
        data_meta = json.load(f)
    # the cache must match this notebook's settings, otherwise resuming would silently mix two data definitions
    expect = {"block": BLOCK, "enc_seq_len": ENC_SEQ_LEN, "dec_seq_len": DEC_SEQ_LEN, "seed": SEED, "val_doc_pct": VAL_DOC_PCT}
    bad = {k: (data_meta.get(k), v) for k, v in expect.items() if data_meta.get(k) != v}
    assert not bad, f"data cache built with different settings (cache, notebook): {bad}"
    assert train_blocks.dtype == np.uint16 and train_blocks.ndim == 2 and train_blocks.shape[1] == BLOCK
    assert val_blocks.dtype == np.uint16 and val_blocks.shape[1] == BLOCK and len(val_blocks) > 0
    assert task_offsets[-1] == len(task_tokens) and len(task_plen) == len(task_offsets) - 1 and len(task_plen) > 0
    assert len(eval_set) > 0
else:
    print("Building data from the official corpus (session 1).")
    rng = random.Random(SEED)
    seen = set()                       # dedup key = MD5 of first 500 chars, global across all sources
    train_tok = array.array('H')
    val_tok = array.array('H')
    task_tok = array.array('H')
    task_off = [0]
    task_pl = []
    eval_set = []
    val_doc_keys = []
    stats = {}
    tokens_per_source = TRAIN_TOKENS // len(TRAIN_FILES)   # explicit equal token budget per source
    val_per_source = VAL_TOKENS // len(TRAIN_FILES)
    task_per_source = TASK_POOL_SIZE // len(TRAIN_FILES)
    eval_per_source = EVAL_SET_SIZE // len(TRAIN_FILES) + 1
    min_words = TASK_PROMPT_WORDS + TASK_TARGET_WORDS

    def doc_blocks(toks):
        n_blocks = len(toks) // BLOCK
        if n_blocks == 0:
            return None
        offset = rng.randint(0, len(toks) - n_blocks * BLOCK)  # don't always start at the heading
        return toks[offset:offset + n_blocks * BLOCK]

    def add_blocks(dst, toks, limit):
        room = (limit - len(dst)) // BLOCK * BLOCK
        if room > 0:
            dst.extend(toks[:room])
        return min(len(toks), max(room, 0)) // BLOCK

    def word_window(words):
        start = rng.randint(0, len(words) - min_words)
        return (" ".join(words[start:start + TASK_PROMPT_WORDS]),
                " ".join(words[start + TASK_PROMPT_WORDS:start + min_words]))

    for train_file in TRAIN_FILES:
        src = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
        print(f"\n--- {src} ---")
        s = stats[src] = Counter()
        blocks_per_doc = []
        file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=train_file,
                                    repo_type="dataset", token=HF_TOKEN)
        train_limit = len(train_tok) + tokens_per_source
        val_limit = len(val_tok) + val_per_source
        task_start, eval_start = len(task_pl), len(eval_set)

        with open(file_path, 'rb') as f:
            for record in ijson.items(f, 'item'):
                s["docs_scanned"] += 1
                text = record.get("text") if isinstance(record, dict) else None
                if not isinstance(text, str):
                    s["malformed_skipped"] += 1
                    continue
                key = hashlib.md5(text[:500].encode()).hexdigest()
                if key in seen:
                    s["dup_dropped"] += 1
                    continue
                seen.add(key)
                words = text.split()
                if len(words) < 50:
                    s["under_50_words_dropped"] += 1
                    continue
                s["docs_kept"] += 1
                is_val = int(key[:8], 16) % 100 < VAL_DOC_PCT

                if is_val:
                    s["val_docs"] += 1
                    val_doc_keys.append(key)
                    if len(eval_set) - eval_start < eval_per_source and len(words) >= min_words:
                        p, t = word_window(words)
                        eval_set.append({"source": src, "doc_key": key, "prompt": p, "reference": t})
                    if len(val_tok) + BLOCK <= val_limit:
                        blocks = doc_blocks(tokenizer.encode(text))
                        if blocks is not None:
                            add_blocks(val_tok, blocks, val_limit)
                else:
                    s["train_docs"] += 1
                    need_blocks = len(train_tok) + BLOCK <= train_limit
                    need_task = len(task_pl) - task_start < task_per_source and len(words) >= min_words
                    if need_blocks:
                        toks = tokenizer.encode(text)
                        s["train_doc_tokens"] += len(toks)
                        blocks = doc_blocks(toks)
                        if blocks is None:
                            s["train_docs_under_1_block"] += 1
                        else:
                            n_added = add_blocks(train_tok, blocks, train_limit)
                            blocks_per_doc.append(n_added)
                    if need_task:
                        p, t = word_window(words)
                        p_ids, t_ids = tokenizer.encode(p), tokenizer.encode(t)
                        if len(p_ids) > ENC_SEQ_LEN - 1:
                            s["task_prompt_truncated"] += 1
                            p_ids = p_ids[-(ENC_SEQ_LEN - 1):]     # same left-truncation as inference
                        if len(t_ids) > DEC_SEQ_LEN:
                            s["task_target_overlength"] += 1
                            t_ids = t_ids[:DEC_SEQ_LEN]
                        task_tok.extend(p_ids)
                        task_tok.extend(t_ids)
                        task_off.append(len(task_tok))
                        task_pl.append(len(p_ids))

                done_blocks = len(train_tok) + BLOCK > train_limit and len(val_tok) + BLOCK > val_limit
                done_pools = (len(task_pl) - task_start >= task_per_source
                              and len(eval_set) - eval_start >= eval_per_source)
                if done_blocks and done_pools:
                    break
                if s["docs_scanned"] % 20000 == 0:
                    print(f"  ... {s['docs_scanned']:,} docs | train={len(train_tok):,} val={len(val_tok):,} "
                          f"task={len(task_pl)-task_start:,} eval={len(eval_set)-eval_start}")
            else:
                s["source_exhausted"] = 1   # ran out of documents before filling every quota

        bpd = np.array(blocks_per_doc) if blocks_per_doc else np.zeros(1)
        s["blocks_from_source"] = int(bpd.sum())
        s["blocks_per_doc_mean"] = float(bpd.mean())
        s["blocks_per_doc_p50"] = float(np.percentile(bpd, 50))
        s["blocks_per_doc_p90"] = float(np.percentile(bpd, 90))
        s["blocks_per_doc_max"] = int(bpd.max())
        top = np.sort(bpd)[::-1]
        s["block_share_top1pct_docs"] = float(top[:max(1, len(top) // 100)].sum() / max(1, top.sum()))
        s["task_examples"] = len(task_pl) - task_start
        s["eval_examples"] = len(eval_set) - eval_start
        gc.collect()
        print(f"  {dict(s)}")

    del seen
    train_blocks = np.frombuffer(train_tok, dtype=np.uint16).reshape(-1, BLOCK).copy()
    val_blocks = np.frombuffer(val_tok, dtype=np.uint16).reshape(-1, BLOCK).copy()
    task_tokens = np.frombuffer(task_tok, dtype=np.uint16).copy()
    task_offsets = np.array(task_off, dtype=np.int64)
    task_plen = np.array(task_pl, dtype=np.int32)
    del train_tok, val_tok, task_tok
    gc.collect()
    eval_set = eval_set[:EVAL_SET_SIZE]

    data_meta = {
        "seed": SEED, "block": BLOCK, "enc_seq_len": ENC_SEQ_LEN, "dec_seq_len": DEC_SEQ_LEN, "val_doc_pct": VAL_DOC_PCT,
        "split_rule": "doc is validation iff int(md5(text[:500])[:8], 16) % 100 < VAL_DOC_PCT",
        "dedup_rule": "drop any doc whose md5(text[:500]) was already seen, across all four sources",
        "source_weighting": f"equal block-token budget per source ({tokens_per_source:,} tokens each)",
        "task_pool": f"one random 512->128 word window per training doc with >= {min_words} words, until the per-source quota",
        "stats": {k: dict(v) for k, v in stats.items()},
        "val_doc_keys": val_doc_keys,
        "eval_doc_keys": [e["doc_key"] for e in eval_set],
    }
    np.save(os.path.join(OUTPUT_DIR, CACHE["train"]), train_blocks)
    np.save(os.path.join(OUTPUT_DIR, CACHE["val"]), val_blocks)
    np.savez(os.path.join(OUTPUT_DIR, CACHE["task"]), tokens=task_tokens, offsets=task_offsets, plen=task_plen)
    with open(os.path.join(OUTPUT_DIR, CACHE["eval"]), "w") as f:
        json.dump(eval_set, f)
    with open(os.path.join(OUTPUT_DIR, CACHE["meta"]), "w") as f:
        json.dump(data_meta, f)

# ---- audit report ----
print(f"\n{'='*60}\nDATA AUDIT")
for src, s in data_meta["stats"].items():
    print(f"{src:14s} scanned={s.get('docs_scanned',0):>8,} kept={s.get('docs_kept',0):>8,} "
          f"dup={s.get('dup_dropped',0):>6,} <50w={s.get('under_50_words_dropped',0):>6,} "
          f"val_docs={s.get('val_docs',0):>6,} blocks={s.get('blocks_from_source',0):>8,} "
          f"<1block={s.get('train_docs_under_1_block',0):>7,} exhausted={bool(s.get('source_exhausted'))}")
    print(f"{'':14s} blocks/doc mean={s.get('blocks_per_doc_mean',0):.1f} p50={s.get('blocks_per_doc_p50',0):.0f} "
          f"p90={s.get('blocks_per_doc_p90',0):.0f} max={s.get('blocks_per_doc_max',0)} "
          f"top-1%-docs share={s.get('block_share_top1pct_docs',0):.1%} | task={s.get('task_examples',0):,} "
          f"(prompt trunc={s.get('task_prompt_truncated',0)}, target overlength={s.get('task_target_overlength',0)}) "
          f"eval={s.get('eval_examples',0)}")
n_task = len(task_plen)
print(f"\nTrain blocks: {len(train_blocks):,} ({len(train_blocks)*BLOCK/1e6:.0f}M tokens) | val blocks: {len(val_blocks):,}")
print(f"Task pool: {n_task:,} examples | eval set: {len(eval_set)} (validation documents only)")
print("Sampling: blocks are drawn uniformly, so a document's share is proportional to its length in blocks "
      "(long documents contribute more blocks; documents under 1024 tokens are excluded from the block pool "
      "but can appear in the task pool). Task examples: at most one per document, independent of length.")
print(f"Source weighting: {data_meta['source_weighting']}")


## 7. Batches: Continuation, Span Corruption, Task-Matched

- **Continuation**: encoder `<cont>` + 703 tokens, decoder target = the next 320 tokens of the same document.
- **Span corruption**: 760 original tokens, ~15% removed in spans of mean length 3, each replaced by its own sentinel; target = `<extra_id_i> span_i ... </s>`.
- **Task-matched**: 512 words of prompt (last 703 tokens kept, as at inference) -> the next 128 words.
- Every optimizer update uses one objective for all its microbatches; samplers and objective choice are resumable.

In [ ]:
tokenizer._encode_cache = {}
gc.collect()


class EpochSampler:
    """Walks a seeded permutation of item indices, reshuffling each epoch. State = (epoch, pos), so it resumes exactly."""
    def __init__(self, n, seed):
        self.n, self.seed, self.epoch, self.pos = n, seed, 0, 0
        self._perm()

    def _perm(self):
        self.perm = np.random.default_rng([self.seed, self.epoch]).permutation(self.n)

    def take(self, k):
        out = []
        while len(out) < k:
            if self.pos >= self.n:
                self.epoch += 1
                self.pos = 0
                self._perm()
            m = min(k - len(out), self.n - self.pos)
            out.extend(self.perm[self.pos:self.pos + m].tolist())
            self.pos += m
        return np.array(out, dtype=np.int64)

    def state(self):
        return {"epoch": self.epoch, "pos": self.pos}

    def load(self, st):
        self.epoch, self.pos = st["epoch"], st["pos"]
        self._perm()


block_sampler = EpochSampler(len(train_blocks), SEED + 1)   # shared by continuation and span corruption
task_sampler = EpochSampler(len(task_plen), SEED + 2)


def objective_for(update_idx, final_start):
    """Objective of an optimizer update: a pure function of (seed, update index), so it is reproducible and resumable.
    Every microbatch of one update uses the same objective (homogeneous batches)."""
    if final_start is not None and update_idx >= final_start:
        return "task"
    r = np.random.default_rng([SEED, 7, update_idx]).random()
    return "denoise" if r < P_DENOISE else "cont"


def span_corrupt(tokens, rng):
    """T5-style span corruption. ~15% of tokens removed in spans of mean length 3; each removed span is replaced by
    its own sentinel in the input, and the target lists <extra_id_i> span_i ... followed by EOS."""
    L = len(tokens)
    n_noise = min(max(int(round(L * SPAN_NOISE_DENSITY)), 1), L - 1)
    n_spans = min(max(int(round(n_noise / SPAN_MEAN_LEN)), 1), N_SENTINELS, L - n_noise)

    def partition(total, n):  # random split of total into n positive parts
        if n == 1:
            return np.array([total])
        cuts = np.sort(rng.choice(np.arange(1, total), n - 1, replace=False))
        return np.diff(np.concatenate([[0], cuts, [total]]))

    noise_lens = partition(n_noise, n_spans)
    keep_lens = partition(L - n_noise, n_spans)
    inp, tgt, p = [DENOISE_ID], [], 0
    for i in range(n_spans):
        inp.extend(tokens[p:p + keep_lens[i]]); p += keep_lens[i]
        inp.append(SENTINEL_IDS[i]); tgt.append(SENTINEL_IDS[i])
        tgt.extend(tokens[p:p + noise_lens[i]]); p += noise_lens[i]
    tgt.append(EOS_ID)
    return inp, tgt


def collate(encs, tgts):
    """Right-pad a list of encoder inputs and targets. Decoder input = BOS + target shifted right."""
    B = len(encs)
    Te = max(len(e) for e in encs)
    Td = max(len(t) for t in tgts)
    assert Te <= ENC_SEQ_LEN and Td <= DEC_SEQ_LEN, (Te, Td)
    enc = np.full((B, Te), PAD_ID, dtype=np.int64)
    enc_mask = np.zeros((B, Te), dtype=bool)
    dec_in = np.full((B, Td), PAD_ID, dtype=np.int64)
    tgt = np.full((B, Td), IGNORE_INDEX, dtype=np.int64)
    for i, (e, t) in enumerate(zip(encs, tgts)):
        enc[i, :len(e)] = e
        enc_mask[i, :len(e)] = True
        dec_in[i, 0] = BOS_ID
        dec_in[i, 1:len(t)] = t[:-1]
        tgt[i, :len(t)] = t
    return enc, enc_mask, dec_in, tgt


def make_microbatch(obj, update_idx, micro_idx, blocks=None, samplers=None):
    """One homogeneous microbatch of BATCH_SIZE examples for objective obj.
    blocks/samplers default to the training data; pass val data for validation."""
    blocks = train_blocks if blocks is None else blocks
    bs, ts = (block_sampler, task_sampler) if samplers is None else samplers
    if obj == "cont":
        # contiguous text from one document: encoder = <cont> + 703 tokens, decoder target = the next 320
        blk = blocks[bs.take(BATCH_SIZE)].astype(np.int64)
        encs = [np.concatenate([[CONT_ID], b[:ENC_SEQ_LEN - 1]]) for b in blk]
        tgts = [b[ENC_SEQ_LEN - 1:ENC_SEQ_LEN - 1 + DEC_SEQ_LEN] for b in blk]
    elif obj == "denoise":
        blk = blocks[bs.take(BATCH_SIZE)].astype(np.int64)
        encs, tgts = [], []
        for b_i, b in enumerate(blk):
            rng = np.random.default_rng([SEED, 11, update_idx, micro_idx, b_i])
            e, t = span_corrupt(b[:SPAN_SRC_LEN].tolist(), rng)
            encs.append(e); tgts.append(t)
    elif obj == "task":
        idx = ts.take(BATCH_SIZE)
        encs, tgts = [], []
        for j in idx:
            a, z, pl = task_offsets[j], task_offsets[j + 1], task_plen[j]
            seq = task_tokens[a:z].astype(np.int64)
            encs.append(np.concatenate([[CONT_ID], seq[:pl]]))
            tgts.append(seq[pl:])
    else:
        raise ValueError(obj)
    return collate(encs, tgts)


def to_device(mb):
    return tuple(torch.from_numpy(np.ascontiguousarray(a)).to(device, non_blocking=True) for a in mb)


# quick shape report
_probe = EpochSampler(len(train_blocks), 999), EpochSampler(len(task_plen), 999)
for _obj in ("cont", "denoise", "task"):
    e, m, d, t = make_microbatch(_obj, 0, 0, samplers=_probe)
    print(f"{_obj:8s} enc {e.shape} (real {m.sum(1).tolist()}) dec {d.shape} "
          f"target tokens {(t != IGNORE_INDEX).sum(1).tolist()}")
print("32 examples per update = 32 x (704 + 320) = 32,768 positions only for full continuation batches; "
      "span-corruption (~685 + ~153) and task-matched batches (padded, ~556-704 + ~139-202) process fewer.")


## 8. Model Architecture: RoPE and Attention

- **rope.py**: `build_rope_cache`, `rotate_half`, `apply_rope` from A3 Q3.1 (split-half convention)
- **CausalSelfAttention**: decoder self-attention, the A3 Q3 class (RoPE on q/k, MHA)
- **SelfAttention**: encoder self-attention, the same class without the causal mask, plus a key-padding mask
- **CrossAttention**: new; decoder queries attend to encoder keys/values, full MHA, no RoPE, encoder padding masked

In [ ]:
%%writefile rope.py
"""Rotary Positional Embeddings, implemented from scratch (from A3 Q3.1)."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"

    # this is done so that we can smoothly split our dimensions correctly the way we want to

    inverse_frequencies = torch.zeros(d_head//2) # (d_head/2,)

    for i in range(0,d_head//2,1):

        inverse_frequencies[i] = (1 / (base ** (2*i / d_head))) # theta_i for all i pairs



    phi_values = torch.zeros(max_seq_len, d_head//2)
    positions = [i for i in range(0,max_seq_len)]

    for i in range(len(positions)):
        for j in range(d_head//2):
            phi_values[i,j] = positions[i] * inverse_frequencies[j]  # where positions[i] is a scalar that is being multed with another scalar value from inmverse frequencies

    # finally makes phi_values a T x d_head/2 tensor


    angles = torch.cat([phi_values,phi_values], dim=-1)  # can do this because in the phi matrix, column 0 and column d_head/2 will have same values as they have same value of i (i.e. their pair index)

    return angles.cos(), angles.sin() # sin and cos matrices



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""

    # a 90 degree rotation of (e,a) is (-a,e)
    last_dim = x.shape[-1]
    half = last_dim // 2

    x1 = x[..., :half] # if x has dimension A,B,C,D then x1 will have dimension A,B,C,D/2 and will store values of all AxBxC configurations but only D/2 values of it... like [1,2,3,4] and then  5,6,7,8 below it, so x1 will have 1,2 and below it 5,6 # just halve it along the last dim
    x2 = x[..., half:]


    ans = torch.cat([-x2, x1], dim=-1) # x rotated by 90 degree no mattewr whgat the dimension is
    return ans


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    T, d_head = cos.shape
    cos = cos.view(1,1,T,d_head)
    sin = sin.view(1,1,T,d_head)

    x_rotated = x * cos + rotate_half(x)*sin # applying rotate_half and adding just is equivalent to rotating the entire vector by theta which is what we do in rope

    return x_rotated

In [ ]:
%%writefile attention.py
"""Attention modules for the Encoder-Decoder Transformer.
CausalSelfAttention is the A3 Q3 class (MHA/GQA/MQA + optional RoPE).
SelfAttention is the same class with the causal mask removed (bidirectional encoder) and a key-padding mask.
CrossAttention is new: decoder queries attend to encoder keys/values, no RoPE, key-padding mask on the encoder side.
Decoder padding is only ever at the end of a sequence, so the causal mask already stops real tokens attending to it."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2,-1))*self.attention_scale
        scores = scores.masked_fill(self.causal_mask[:T,:T] == 0, float('-inf'))
        probs = torch.softmax(scores, dim=-1)

        probs = self.attn_dropout(probs)
        out = probs @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)

        return out


class SelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder: the A3 Q3 CausalSelfAttention without the causal mask."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, key_padding_mask=None):
        """key_padding_mask: (B, T) bool, True = real token."""
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2,-1))*self.attention_scale
        # no causal mask: every prompt token sees the whole prompt; padded keys are masked out
        if key_padding_mask is not None:
            scores = scores.masked_fill(~key_padding_mask[:, None, None, :], float('-inf'))
        probs = torch.softmax(scores, dim=-1)

        probs = self.attn_dropout(probs)
        out = probs @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)

        return out


class CrossAttention(nn.Module):
    """Decoder queries attend to encoder keys/values (full MHA).
    No RoPE: encoder position i and decoder position i do not share a coordinate system."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, enc_output: torch.Tensor, enc_mask=None) -> torch.Tensor:
        """enc_mask: (B, T_enc) bool, True = real encoder token."""
        B, T_dec, C = x.shape
        T_enc = enc_output.shape[1]

        q = self.q_proj(x) # query from decoder
        k = self.k_proj(enc_output) # key from encoder
        v = self.v_proj(enc_output) # value from encoder

        q = q.view(B, T_dec, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T_enc, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T_enc, self.n_heads, self.d_head).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if enc_mask is not None:
            scores = scores.masked_fill(~enc_mask[:, None, None, :], float('-inf'))
        A = torch.softmax(scores, dim=-1)
        A = self.attn_dropout(A)
        out = A @ v

        out = out.transpose(1, 2).contiguous().view(B, T_dec, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


## 9. Model Architecture: Encoder-Decoder LM

- **RMSNorm** and **SwiGLU**: new (not part of A3), standard LLaMA-style definitions
- **EncoderBlock / DecoderBlock / EncoderDecoderLM**: new, pre-norm residual blocks composed from the modules above
- Final RMSNorm on the encoder output so cross-attention reads a normalised memory (pre-norm stacks otherwise leave the residual stream unnormalised)

In [ ]:
%%writefile model_arch.py
"""Encoder-Decoder Transformer LM: RMSNorm pre-norm, SwiGLU FFN, RoPE in self-attention,
full multi-head cross-attention, one shared embedding matrix (encoder, decoder, output head)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import SelfAttention, CausalSelfAttention, CrossAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_enc_layers: int = 10
    n_dec_layers: int = 4
    n_heads: int = 10
    d_ff: int = 1712
    max_enc_len: int = 704
    max_dec_len: int = 320
    dropout: float = 0.0


class RMSNorm(nn.Module):
    """x / RMS(x) * g. No mean-centering and no bias, unlike LayerNorm.
    The mean-square is computed in fp32 so x*x cannot overflow under fp16 autocast."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return xf.type_as(x) * self.weight


class SwiGLU(nn.Module):
    """down( SiLU(gate(x)) * up(x) ). Three matrices, so d_ff ~ 2/3 * 4 * d_model keeps
    the parameter count equal to a 4*d_model GELU FFN."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class EncoderBlock(nn.Module):
    """Pre-norm: x + Attn(RMSNorm(x)), then x + SwiGLU(RMSNorm(x)). Bidirectional attention with RoPE."""
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = RMSNorm(cfg.d_model)
        self.attn = SelfAttention(cfg.d_model, cfg.n_heads, cfg.n_heads, cfg.max_enc_len,
                                  cfg.dropout, use_rope=True)
        self.ln2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, enc_mask=None):
        x = x + self.attn(self.ln1(x), enc_mask)
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Pre-norm: causal self-attn (RoPE) + cross-attn to encoder + SwiGLU."""
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = RMSNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_heads, cfg.max_dec_len,
                                             cfg.dropout, use_rope=True)
        self.ln2 = RMSNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln3 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, enc_output, enc_mask=None):
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_output, enc_mask)
        x = x + self.ffn(self.ln3(x))
        return x


class EncoderDecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)  # shared by encoder and decoder
        self.enc_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_enc_layers)])
        self.enc_norm = RMSNorm(cfg.d_model)
        self.dec_blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_dec_layers)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def encode(self, enc_input, enc_mask=None):
        x = self.tok_emb(enc_input)
        for block in self.enc_blocks:
            x = block(x, enc_mask)
        return self.enc_norm(x)

    def decode(self, dec_input, enc_output, enc_mask=None):
        x = self.tok_emb(dec_input)
        for block in self.dec_blocks:
            x = block(x, enc_output, enc_mask)
        x = self.ln_f(x)
        return self.head(x)

    def forward(self, enc_input, dec_input, enc_mask=None):
        enc_output = self.encode(enc_input, enc_mask)
        return self.decode(dec_input, enc_output, enc_mask)


## 10. Initialize Model

In [ ]:
import torch.nn as nn
import importlib, sys
for _m in ("model_arch", "attention", "rope"):
    sys.modules.pop(_m, None)
from model_arch import LMConfig, EncoderDecoderLM, RMSNorm

cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_enc_layers=MODEL_ENC_LAYERS,
    n_dec_layers=MODEL_DEC_LAYERS,
    n_heads=MODEL_HEADS,
    d_ff=MODEL_FF,
    max_enc_len=MAX_ENC_LEN,
    max_dec_len=MAX_DEC_LEN,
    dropout=MODEL_DROPOUT,
)

torch.manual_seed(SEED)
model = EncoderDecoderLM(cfg).to(device)   # parameters stay fp32; autocast handles fp16 compute

def init_weights(module):
    if module is model.head:
        return  # head.weight is the tied embedding matrix; initialise it once, via tok_emb
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, RMSNorm):
        torch.nn.init.ones_(module.weight)

model.apply(init_weights)

# GPT-2-style heuristic: projections that write into the residual stream get std 0.02 / sqrt(2 * total_layers)
# with total_layers = 10 + 4 = 14. A heuristic carried over from decoder-only practice, not a derived optimum.
total_layers = MODEL_ENC_LAYERS + MODEL_DEC_LAYERS
residual_std = 0.02 / math.sqrt(2 * total_layers)
residual_proj = []
for block in model.enc_blocks:
    residual_proj += [block.attn.out_proj, block.ffn.down_proj]
for block in model.dec_blocks:
    residual_proj += [block.self_attn.out_proj, block.cross_attn.out_proj, block.ffn.down_proj]
for lin in residual_proj:
    torch.nn.init.normal_(lin.weight, mean=0.0, std=residual_std)

# ---- init audit ----
norms = [m for m in model.modules() if isinstance(m, RMSNorm)]
biases = [n for n, p in model.named_parameters() if n.endswith(".bias")]
assert model.head.weight is model.tok_emb.weight, "weight tying broken"
assert all(torch.all(m.weight == 1) for m in norms)
assert all(p.dtype == torch.float32 for p in model.parameters())
n_params = sum(p.numel() for p in model.parameters())
print(f"RMSNorm modules: {len(norms)} (all scales = 1) | Linear biases: {len(biases)} | "
      f"residual-scaled projections: {len(residual_proj)} at std {residual_std:.5f} "
      f"({2*MODEL_ENC_LAYERS} encoder + {3*MODEL_DEC_LAYERS} decoder incl. {MODEL_DEC_LAYERS} cross-attention out_proj)")
print(f"Tied embedding: head.weight is tok_emb.weight -> {model.head.weight is model.tok_emb.weight}, "
      f"shape {tuple(model.tok_emb.weight.shape)}")
print(f"Model parameters: {n_params:,} ({n_params/1e6:.1f}M)")
assert 50e6 <= n_params <= 300e6, "outside the 50-300M parameter rule"

use_amp = torch.cuda.is_available()
def amp_ctx():
    return torch.autocast(device_type="cuda" if use_amp else "cpu", dtype=torch.float16, enabled=use_amp)

def GradScaler(device_type="cuda", enabled=True):
    """torch.amp.GradScaler on PyTorch >= 2.3, torch.cuda.amp.GradScaler (as in Arch 1/2) otherwise."""
    try:
        from torch.amp import GradScaler as _GS
        return _GS(device_type, enabled=enabled)
    except (ImportError, TypeError):
        from torch.cuda.amp import GradScaler as _GS
        return _GS(enabled=enabled)
print(f"torch {torch.__version__} | fp16 autocast: {use_amp}")


## 11. Optimizer with Weight Decay Groups

In [ ]:
# weight decay only on 2D params (Linear weights), NOT on biases + LayerNorm
decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if param.dim() >= 2:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

print(f"Decay params:    {sum(p.numel() for p in decay_params):,}")
print(f"No-decay params: {sum(p.numel() for p in no_decay_params):,}")

optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8)

## 12. Learning Rate Schedule

500 warmup updates, then cosine 6e-4 -> 6e-5 over a planned horizon that is measured once (updates 50-300, all objectives) and frozen for the whole run.

In [ ]:
assert MEASURE_TO <= WARMUP_STEPS, "the horizon must be measured inside warmup, before cosine decay starts"

def lr_at(update, planned_total):
    """Linear warmup over WARMUP_STEPS successful updates, then cosine from PEAK_LR to MIN_LR across the whole planned run.
    planned_total is measured once (updates MEASURE_FROM..MEASURE_TO, inside warmup) and then frozen for the run."""
    if update < WARMUP_STEPS:
        return PEAK_LR * update / WARMUP_STEPS
    assert planned_total is not None, "update horizon must be fixed before warmup ends"
    progress = min(1.0, (update - WARMUP_STEPS) / max(1, planned_total - WARMUP_STEPS))
    return MIN_LR + 0.5 * (1.0 + math.cos(math.pi * progress)) * (PEAK_LR - MIN_LR)


def plan_horizon(tp):
    """Planned total updates from measured seconds per update of each objective, fitted to TRAIN_BUDGET_HOURS."""
    t_cont = float(np.mean(tp["cont"]))
    t_den = float(np.mean(tp["denoise"])) if tp["denoise"] else t_cont
    t_task = t_cont  # task batches are at most as long as continuation batches; conservative
    per_update = (1 - FINAL_TASK_FRAC) * ((1 - P_DENOISE) * t_cont + P_DENOISE * t_den) + FINAL_TASK_FRAC * t_task
    total = int(TRAIN_BUDGET_HOURS * 3600 * HORIZON_SAFETY / per_update)
    final_start = int(total * (1 - FINAL_TASK_FRAC))
    return total, final_start, {"s_per_update_cont": t_cont, "s_per_update_denoise": t_den,
                                "s_per_update_blended": per_update}

print(f"Warmup {WARMUP_STEPS} updates, cosine {PEAK_LR} -> {MIN_LR} over a horizon measured during warmup "
      f"(updates {MEASURE_FROM}-{MEASURE_TO}) and fitted to {TRAIN_BUDGET_HOURS}h of active training.")


## 13. Generation and Held-out Evaluation

Used by the training loop at milestones (mid-run, before the task-matched segment, end) and for the test predictions.

In [ ]:
import torch.nn.functional as F

NEVER_GEN_T = torch.tensor(NEVER_GENERATE, device=device)


def generate(model, tokenizer, prompt, max_tokens=MAX_GEN_TOKENS, target_words=TARGET_WORDS,
             temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, repetition_penalty=REPETITION_PENALTY,
             seed=None, return_info=False):
    """Continuation of `prompt` (only the continuation is returned, never the prompt or special tokens).
    Encoder input = <cont> + the last 703 prompt tokens (same truncation as the task-matched training data).
    Special tokens (pad/unk/bos/eos/mode/sentinels) can never be sampled, so there is no early EOS.
    Stops once `target_words` complete words exist; if the token cap is hit first, returns fewer words (no filler)."""
    model.eval()
    p_ids = tokenizer.encode(prompt)
    prompt_truncated = len(p_ids) > ENC_SEQ_LEN - 1
    p_ids = p_ids[-(ENC_SEQ_LEN - 1):]
    enc_input = torch.tensor([[CONT_ID] + p_ids], dtype=torch.long, device=device)
    max_tokens = min(max_tokens, MAX_DEC_LEN - 1)
    gen = None
    if temperature > 0:
        gen = torch.Generator(device=device)
        gen.manual_seed(SEED if seed is None else int(seed))

    with torch.no_grad():
        with amp_ctx():
            enc_out = model.encode(enc_input)
        dec, generated, complete = [BOS_ID], [], False
        for _ in range(max_tokens):
            dec_input = torch.tensor([dec], dtype=torch.long, device=device)
            with amp_ctx():
                logits = model.decode(dec_input, enc_out)
            logits = logits[0, -1].float()
            logits[NEVER_GEN_T] = float('-inf')

            if repetition_penalty != 1.0 and generated:
                ids = torch.tensor(sorted(set(generated[-100:])), device=device)
                vals = logits[ids]
                # penalise in the right direction for both signs (dividing a negative logit would raise it)
                logits[ids] = torch.where(vals > 0, vals / repetition_penalty, vals * repetition_penalty)

            if temperature == 0.0:
                tok = int(logits.argmax())
            else:
                logits = logits / temperature
                if top_k > 0:
                    kth = torch.topk(logits, min(top_k, logits.size(-1))).values[-1]
                    logits[logits < kth] = float('-inf')
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                remove = cum > top_p
                remove[1:] = remove[:-1].clone()
                remove[0] = False
                logits[sorted_idx[remove]] = float('-inf')
                probs = torch.softmax(logits, dim=-1)
                tok = int(torch.multinomial(probs, 1, generator=gen))

            dec.append(tok)
            generated.append(tok)
            n_words = len(tokenizer.decode(generated).split())
            if n_words > target_words or (n_words == target_words and tokenizer._ends_word.get(tok, False)):
                complete = True
                break

    words = tokenizer.decode(generated).split()[:target_words]
    text = " ".join(words)
    info = {"n_words": len(words), "n_tokens": len(generated), "hit_token_cap": not complete,
            "prompt_truncated": prompt_truncated}
    return (text, info) if return_info else text


def val_loss_cont(max_batches=None):
    """Teacher-forced continuation loss on validation blocks (same batches every call)."""
    was_training = model.training
    model.eval()
    samp = (EpochSampler(len(val_blocks), SEED + 3), EpochSampler(1, 0))
    n_batches = len(val_blocks) // BATCH_SIZE if max_batches is None else min(max_batches, len(val_blocks) // BATCH_SIZE)
    tot, ntok = 0.0, 0
    with torch.no_grad():
        for b in range(n_batches):
            enc, enc_mask, dec_in, tgt = to_device(make_microbatch("cont", 0, b, blocks=val_blocks, samplers=samp))
            with amp_ctx():
                logits = model(enc, dec_in, enc_mask)
            tot += F.cross_entropy(logits.float().view(-1, logits.size(-1)), tgt.view(-1),
                                   ignore_index=IGNORE_INDEX, reduction="sum").item()
            ntok += int((tgt != IGNORE_INDEX).sum())
    if was_training:
        model.train()
    return tot / max(1, ntok)


def repetition_rate(text, n=4):
    w = text.split()
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not grams else 1.0 - len(set(grams)) / len(grams)


def evaluate_heldout(tag):
    """Generate for the fixed held-out set (validation documents only) and score with BERTScore.
    BERTScore here = bert_score package, roberta-large defaults (layer 17, no idf, no baseline rescaling).
    The official scoring script/settings are not available in this notebook, so treat this as an approximation."""
    from bert_score import BERTScorer   # installed and test-loaded in the pre-flight cell

    t0 = time.time()
    preds, infos, errors = [], [], []
    for i, ex in enumerate(eval_set):
        try:
            text, info = generate(model, tokenizer, ex["prompt"], seed=SEED + i, return_info=True)
        except Exception as e:  # report, never substitute filler
            text, info = "", {"n_words": 0, "n_tokens": 0, "hit_token_cap": False, "prompt_truncated": False}
            errors.append({"idx": i, "error": repr(e)})
        preds.append(text)
        infos.append(info)
    refs = [ex["reference"] for ex in eval_set]
    scorer = BERTScorer(model_type="roberta-large", lang="en", device=str(device))
    P, R, F1 = scorer.score(preds, refs)
    del scorer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    vl = val_loss_cont(MAX_VAL_BATCHES)
    res = {
        "tag": tag, "update": run["update"], "bertscore_P": P.mean().item(), "bertscore_R": R.mean().item(),
        "bertscore_F1": F1.mean().item(), "val_cont_loss": vl, "val_cont_ppl": math.exp(min(vl, 20)),
        "mean_words": float(np.mean([x["n_words"] for x in infos])),
        "short_outputs": sum(x["n_words"] < TARGET_WORDS for x in infos),
        "token_cap_hits": sum(x["hit_token_cap"] for x in infos),
        "prompt_truncated": sum(x["prompt_truncated"] for x in infos),
        "repetition_4gram": float(np.mean([repetition_rate(p) for p in preds])),
        "generation_errors": len(errors), "n": len(eval_set), "seconds": time.time() - t0,
        "per_source_F1": {s: float(np.mean([F1[i].item() for i, ex in enumerate(eval_set) if ex["source"] == s]))
                          for s in sorted({ex["source"] for ex in eval_set})},
    }
    with open(os.path.join(OUTPUT_DIR, f"eval_{tag}.json"), "w") as f:
        json.dump({"result": res, "errors": errors, "predictions": preds}, f)
    print(f"  >> EVAL [{tag}] update {res['update']}: BERTScore F1={res['bertscore_F1']:.4f} "
          f"(P={res['bertscore_P']:.4f} R={res['bertscore_R']:.4f}) | val cont loss={vl:.4f} | "
          f"short={res['short_outputs']} cap_hits={res['token_cap_hits']} rep4={res['repetition_4gram']:.3f} "
          f"errors={len(errors)} | {res['seconds']/60:.1f} min")
    model.train()
    return res


## 14. Pre-flight Checks

Fast functional checks on the real model and data before the long training stretch. Run on the T4, check 11 also exercises the fp16 path.

In [ ]:
import copy
import torch.nn.functional as F

def check(name, cond, detail=""):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
    assert cond, name

print(f"Running pre-flight checks on {device} (fp16 autocast: {use_amp})\n")

# 1. continuation batch: shift, contiguity, no target leakage into the encoder
s_a, s_b = EpochSampler(len(train_blocks), 4242), EpochSampler(len(train_blocks), 4242)
enc, m, dec_in, tgt = make_microbatch("cont", 0, 0, samplers=(s_a, EpochSampler(1, 0)))
blk = train_blocks[s_b.take(BATCH_SIZE)].astype(np.int64)
check("cont: encoder = <cont> + block[0:703]", np.all(enc[:, 0] == CONT_ID) and np.array_equal(enc[:, 1:], blk[:, :ENC_SEQ_LEN - 1]))
check("cont: target = block[703:1023], the tokens right after the prompt (so the encoder never sees them)",
      np.array_equal(tgt, blk[:, ENC_SEQ_LEN - 1:ENC_SEQ_LEN - 1 + DEC_SEQ_LEN]))
check("cont: decoder input = BOS + target[:-1]", np.all(dec_in[:, 0] == BOS_ID) and np.array_equal(dec_in[:, 1:], tgt[:, :-1]))

# 2. task batch: shift with padding, mask, loss positions
enc, m, dec_in, tgt = make_microbatch("task", 0, 0, samplers=(EpochSampler(1, 0), EpochSampler(len(task_plen), 7)))
ok = True
for i in range(BATCH_SIZE):
    n = int((tgt[i] != IGNORE_INDEX).sum())
    ok &= dec_in[i, 0] == BOS_ID and np.array_equal(dec_in[i, 1:n], tgt[i, :n - 1]) and np.all(tgt[i, n:] == IGNORE_INDEX)
    ok &= enc[i, 0] == CONT_ID and m[i].sum() == (enc[i] != PAD_ID).sum()
check("task: shifted decoder input, padded targets ignored, encoder mask matches real tokens", bool(ok),
      f"target lens {(tgt != IGNORE_INDEX).sum(1).tolist()}")

# 3. span corruption reconstructs the original and fits the limits
lens_in, lens_t, noise_frac, span_lens = [], [], [], []
rec_ok = True
for k in range(300):
    src = train_blocks[k % len(train_blocks), :SPAN_SRC_LEN].astype(np.int64).tolist()
    inp, t = span_corrupt(src, np.random.default_rng([1, k]))
    lens_in.append(len(inp)); lens_t.append(len(t))
    sent_pos = [j for j, x in enumerate(t) if x in SENTINEL_IDS]
    spans = [t[a + 1:b] for a, b in zip(sent_pos, sent_pos[1:] + [len(t) - 1])]
    rebuilt, si = [], 0
    for x in inp[1:]:
        if x in SENTINEL_IDS:
            rec_ok &= x == SENTINEL_IDS[si]
            rebuilt.extend(spans[si]); si += 1
        else:
            rebuilt.append(x)
    rec_ok &= rebuilt == src and inp[0] == DENOISE_ID and t[-1] == EOS_ID and [t[j] for j in sent_pos] == SENTINEL_IDS[:len(sent_pos)]
    noise_frac.append(sum(len(s) for s in spans) / len(src)); span_lens += [len(s) for s in spans]
check("span corruption: input + target rebuild the original exactly, sentinels in order, EOS terminal", bool(rec_ok))
check("span corruption: fits encoder/decoder limits", max(lens_in) <= ENC_SEQ_LEN and max(lens_t) <= DEC_SEQ_LEN,
      f"input {min(lens_in)}-{max(lens_in)}, target {min(lens_t)}-{max(lens_t)}, noise {np.mean(noise_frac):.3f}, "
      f"mean span {np.mean(span_lens):.2f}")

# 4. special IDs and weight tying
specials = [CONT_ID, DENOISE_ID] + SENTINEL_IDS
check("special IDs distinct, appended after the BPE vocab, inside the embedding",
      len(set(specials)) == len(specials) and min(specials) >= BASE_VOCAB and max(specials) < VOCAB_SIZE == model.tok_emb.num_embeddings)
check("tokenizer never emits special IDs on real prompts", max(max(tokenizer.encode(e["prompt"])) for e in eval_set[:20]) < BASE_VOCAB)
check("output head tied to the shared embedding", model.head.weight is model.tok_emb.weight
      and model.head.weight.data_ptr() == model.tok_emb.weight.data_ptr())

# fp32 copy of the real model for the functional checks (does not touch the training model)
mc = copy.deepcopy(model).float().eval()
g = torch.Generator().manual_seed(0)
def rand_ids(*shape):
    return torch.randint(4, BASE_VOCAB, shape, generator=g).to(device)

# 5. causality: changing future decoder tokens cannot change earlier logits
e_in, d_in = rand_ids(2, 48), rand_ids(2, 32)
with torch.no_grad():
    a = mc(e_in, d_in)
    d2 = d_in.clone(); d2[:, 20:] = rand_ids(2, 12)
    b = mc(e_in, d2)
check("decoder causality: logits before the edited position unchanged", torch.allclose(a[:, :20], b[:, :20], atol=1e-4),
      f"max diff {float((a[:, :20] - b[:, :20]).abs().max()):.2e}")

# 6. padding: content of padded encoder positions cannot change any decoder logit
mask = torch.ones(2, 48, dtype=torch.bool, device=device); mask[:, 36:] = False
e_a = e_in.clone(); e_a[:, 36:] = PAD_ID
e_b = e_in.clone(); e_b[:, 36:] = rand_ids(2, 12)
with torch.no_grad():
    la, lb = mc(e_a, d_in, mask), mc(e_b, d_in, mask)
    ea, eb = mc.encode(e_a, mask), mc.encode(e_b, mask)
check("encoder key-padding + cross-attention masks", torch.allclose(la, lb, atol=1e-4) and torch.allclose(ea[:, :36], eb[:, :36], atol=1e-4),
      f"max diff {float((la - lb).abs().max()):.2e}")

# 7. loss excludes padding and accumulation is normalised by total valid target tokens
mc.train()
mbs = [make_microbatch("task", 0, j, samplers=(EpochSampler(1, 0), EpochSampler(len(task_plen), 11 + j))) for j in range(2)]
mbs = [(e[:2, :96], mm[:2, :96], d[:2, :40], t[:2, :40]) for e, mm, d, t in mbs]          # keep it cheap
mbs[1] = (mbs[1][0], mbs[1][1], mbs[1][2], np.where(np.arange(40)[None, :] < np.array([[13], [27]]), mbs[1][3], IGNORE_INDEX))
n_valid = sum(int((t != IGNORE_INDEX).sum()) for *_, t in mbs)
mc.zero_grad()
for mb in mbs:
    e, mm, d, t = to_device(mb)
    (F.cross_entropy(mc(e, d, mm).view(-1, VOCAB_SIZE), t.view(-1), ignore_index=IGNORE_INDEX, reduction="sum") / n_valid).backward()
g_acc = {n: p.grad.clone() for n, p in mc.named_parameters() if p.grad is not None}
mc.zero_grad()
e = torch.cat([to_device(mb)[0] for mb in mbs]); mm = torch.cat([to_device(mb)[1] for mb in mbs])
d = torch.cat([to_device(mb)[2] for mb in mbs]); t = torch.cat([to_device(mb)[3] for mb in mbs])
logits = mc(e, d, mm)
F.cross_entropy(logits.view(-1, VOCAB_SIZE), t.view(-1), ignore_index=IGNORE_INDEX).backward()  # mean over valid tokens
max_gdiff = max(float((p.grad - g_acc[n]).abs().max()) for n, p in mc.named_parameters() if p.grad is not None)
check("accumulated loss = mean over all valid target tokens (unequal microbatches)", max_gdiff < 1e-5, f"max grad diff {max_gdiff:.2e}")
manual = F.cross_entropy(logits.view(-1, VOCAB_SIZE)[t.view(-1) != IGNORE_INDEX], t.view(-1)[t.view(-1) != IGNORE_INDEX])
check("ignore_index drops padded targets", torch.allclose(manual, F.cross_entropy(logits.view(-1, VOCAB_SIZE), t.view(-1), ignore_index=IGNORE_INDEX)))

# 8. encoder and cross-attention receive gradients
gn = lambda p: 0.0 if p.grad is None else float(p.grad.norm())
names = {"enc[0].attn.q_proj": mc.enc_blocks[0].attn.q_proj.weight, "enc[-1].ffn.down_proj": mc.enc_blocks[-1].ffn.down_proj.weight,
         "dec[0].cross_attn.k_proj": mc.dec_blocks[0].cross_attn.k_proj.weight, "dec[-1].cross_attn.v_proj": mc.dec_blocks[-1].cross_attn.v_proj.weight,
         "tok_emb (tied)": mc.tok_emb.weight}
check("gradients reach encoder, cross-attention and embeddings", all(gn(p) > 0 and math.isfinite(gn(p)) for p in names.values()),
      ", ".join(f"{k}={gn(p):.2e}" for k, p in names.items()))
del mc, g_acc, logits
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 9. sampler / objective state resumes exactly
s1 = EpochSampler(1000, 5); s1.take(1500); st = s1.state(); nxt = s1.take(50)
s2 = EpochSampler(1000, 5); s2.load(st)
check("sampler state resumes to the identical next indices (across an epoch boundary)", np.array_equal(nxt, s2.take(50)))
check("objective selection is a pure function of the update index",
      [objective_for(u, None) for u in range(200)] == [objective_for(u, None) for u in range(200)],
      f"denoise share in first 10k updates: {np.mean([objective_for(u, None) == 'denoise' for u in range(10000)]):.3f}")

# 10. save/resume on a tiny model: 2 updates + save + reload + 2 updates == 4 uninterrupted updates
def tiny():
    torch.manual_seed(3)
    return EncoderDecoderLM(LMConfig(vocab_size=VOCAB_SIZE, d_model=64, n_enc_layers=2, n_dec_layers=1, n_heads=4,
                                     d_ff=176, max_enc_len=ENC_SEQ_LEN, max_dec_len=DEC_SEQ_LEN)).to(device)
tiny_batches = [to_device((lambda x: (x[0][:2, :64], x[1][:2, :64], x[2][:2, :32], x[3][:2, :32]))(
    make_microbatch("cont", 0, j, samplers=(EpochSampler(len(train_blocks), 50 + j), EpochSampler(1, 0))))) for j in range(4)]
def tiny_step(mdl, opt, mb):
    opt.zero_grad()
    e, mm, d, t = mb
    F.cross_entropy(mdl(e, d, mm).view(-1, VOCAB_SIZE), t.view(-1), ignore_index=IGNORE_INDEX).backward()
    opt.step()
m_full = tiny(); o_full = torch.optim.AdamW(m_full.parameters(), lr=1e-3)
for mb in tiny_batches:
    tiny_step(m_full, o_full, mb)
m_a = tiny(); o_a = torch.optim.AdamW(m_a.parameters(), lr=1e-3)
for mb in tiny_batches[:2]:
    tiny_step(m_a, o_a, mb)
_tmp = os.path.join(OUTPUT_DIR, "_preflight_resume.pt")
atomic_save({"m": m_a.state_dict(), "o": o_a.state_dict()}, _tmp)
m_b = tiny(); o_b = torch.optim.AdamW(m_b.parameters(), lr=1e-3)
_s = torch.load(_tmp, map_location=device, weights_only=False); m_b.load_state_dict(_s["m"]); o_b.load_state_dict(_s["o"])
os.remove(_tmp)
for mb in tiny_batches[2:]:
    tiny_step(m_b, o_b, mb)
rdiff = max(float((p - q).abs().max()) for p, q in zip(m_full.parameters(), m_b.parameters()))
check("save/resume reproduces the uninterrupted trajectory", rdiff < 1e-5, f"max param diff {rdiff:.2e}")
del m_full, m_a, m_b, o_full, o_a, o_b

# 11. real-size model, real fp16 path: first updates are finite and actually change parameters
mc = copy.deepcopy(model).train()
oc = torch.optim.AdamW(mc.parameters(), lr=PEAK_LR)
sc = GradScaler("cuda", enabled=use_amp)
before = mc.dec_blocks[0].cross_attn.q_proj.weight.detach().clone()
skips, last_loss = 0, None
for attempt in range(6):
    oc.zero_grad(set_to_none=True)
    mb = to_device(make_microbatch("cont", 0, attempt, samplers=(EpochSampler(len(train_blocks), 77 + attempt), EpochSampler(1, 0))))
    with amp_ctx():
        lg = mc(mb[0], mb[2], mb[1])
    loss = F.cross_entropy(lg.float().view(-1, VOCAB_SIZE), mb[3].view(-1), ignore_index=IGNORE_INDEX)
    sc.scale(loss).backward()
    sc.unscale_(oc)
    gnorm = float(torch.nn.utils.clip_grad_norm_(mc.parameters(), GRAD_CLIP))
    s0 = sc.get_scale(); sc.step(oc); sc.update()
    if use_amp and sc.get_scale() < s0:
        skips += 1
        continue
    last_loss = float(loss)
    break
moved = float((mc.dec_blocks[0].cross_attn.q_proj.weight - before).abs().max())
check("first real-size update: finite loss and gradient, parameters move",
      last_loss is not None and math.isfinite(last_loss) and math.isfinite(gnorm) and moved > 0,
      f"loss {last_loss}, grad norm {gnorm:.3f}, skipped fp16 steps before success {skips}, max param change {moved:.2e}")
del mc, oc, sc, lg, loss
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"\nGPU memory after checks: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

print("\nAttention is the hand-written matmul/softmax path (no scaled_dot_product_attention, no FlashAttention), "
      "so SDPA mask semantics and backend availability do not apply. Generation has no KV cache (each step re-runs "
      "the 4-layer decoder over the generated prefix), so there is no cached/uncached comparison to make.")
print("CPU runs of these checks do not establish fp16 GPU stability; check 11 is the fp16 one when run on the T4.")

# 12. BERTScore model availability (fail fast here, never in the middle of training)
try:
    try:
        import bert_score  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'bert_score', '-q'], check=True)
    from bert_score import BERTScorer
    _sc = BERTScorer(model_type="roberta-large", lang="en", device=str(device))
    _p, _r, _f = _sc.score(["the court held that the appeal is dismissed"], ["the appeal was dismissed by the court"])
    del _sc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[PASS] BERTScore (roberta-large) loads and scores (sample F1 {_f.item():.3f})")
except Exception as e:
    EVAL_ENABLED = False
    print(f"[WARN] BERTScore unavailable ({e!r}); held-out BERTScore evals are disabled for this session, "
          f"training and continuation-loss validation are unaffected")


## 15. Training (resumable, one optimization trajectory)

In [ ]:
import torch.nn.functional as F
from dataclasses import asdict

OBJS = ("cont", "denoise", "task")

def new_run_state():
    return {
        "update": 0, "attempts": 0, "skipped_updates": 0,
        "planned_total": None, "final_start": None, "horizon_info": None,
        "tp": {"cont": [], "denoise": []},
        "active_train_sec": 0.0, "eval_sec": 0.0,
        "updates": {o: 0 for o in OBJS}, "examples": {o: 0 for o in OBJS},
        "enc_tokens": {o: 0 for o in OBJS}, "dec_loss_tokens": {o: 0 for o in OBJS},
        "history": [], "val_history": [], "evals": [], "eval_failures": [], "sessions": [], "checkpoints": [],
        "pre_final_saved": False, "mid_evals_done": [], "best_phase1": None, "done": False,
    }

def save_checkpoint(name, kind):
    """Full training state. For LATEST_CKPT the previous latest is kept as PREV_CKPT; every write is atomic."""
    run["checkpoints"].append({"name": name, "kind": kind, "update": run["update"], "unix": time.time()})
    state = {
        "kind": kind, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
        "model_config": asdict(cfg), "run": run,
        "samplers": {"block": block_sampler.state(), "task": task_sampler.state()},
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
        "lr_schedule": {"warmup": WARMUP_STEPS, "peak": PEAK_LR, "min": MIN_LR, "type": "cosine",
                        "note": "LR is a pure function of (run['update'], run['planned_total'])"},
        "special_tokens": SPECIAL_TOKENS, "tokenizer_file": "legal_bpe_16384.json",
        "data_cache_files": CACHE, "split_rule": data_meta["split_rule"], "eval_doc_keys": data_meta["eval_doc_keys"],
        "train_settings": {"P_DENOISE": P_DENOISE, "FINAL_TASK_FRAC": FINAL_TASK_FRAC, "BATCH_SIZE": BATCH_SIZE,
                           "GRAD_ACCUM": GRAD_ACCUM, "TRAIN_BUDGET_HOURS": TRAIN_BUDGET_HOURS, "SEED": SEED},
    }
    path = os.path.join(OUTPUT_DIR, name)
    tmp = path + ".tmp"
    torch.save(state, tmp)
    if name == LATEST_CKPT and os.path.exists(path):
        os.replace(path, os.path.join(OUTPUT_DIR, PREV_CKPT))
    os.replace(tmp, path)
    atomic_json({k: v for k, v in run.items() if k != "tp"}, os.path.join(OUTPUT_DIR, "run_log.json"))
    print(f"  >> saved {name} at update {run['update']:,} ({kind})")

def do_eval(tag):
    """Held-out BERTScore milestone. A failure here is logged and training continues."""
    if not EVAL_ENABLED:
        print(f"  >> eval [{tag}] skipped (BERTScore unavailable this session)")
        return
    t0 = time.time()
    try:
        res = evaluate_heldout(tag)
        run["evals"].append(res)
        if run["best_phase1"] is None or res["bertscore_F1"] > run["best_phase1"]["bertscore_F1"]:
            atomic_save(model.state_dict(), os.path.join(OUTPUT_DIR, BEST_PHASE1))
            run["best_phase1"] = {"tag": tag, "update": run["update"], "bertscore_F1": res["bertscore_F1"]}
            print(f"  >> new best Phase 1 checkpoint ({tag})")
    except Exception as e:
        run["eval_failures"].append({"tag": tag, "update": run["update"], "error": repr(e)})
        print(f"  !! eval [{tag}] failed ({e!r}); training continues")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    finally:
        model.train()
        run["eval_sec"] += time.time() - t0

# ---------------- optimizer state / resume ----------------
scaler = GradScaler("cuda", enabled=use_amp)
run = new_run_state()

if RESUME_DIR and os.path.abspath(RESUME_DIR) != os.path.abspath(OUTPUT_DIR):
    # carry earlier artifacts into this session's output so the next session's input has everything
    for name in (FOUNDATION_CKPT, BEST_PHASE1, PREV_CKPT):
        src = os.path.join(RESUME_DIR, name)
        if os.path.exists(src) and not os.path.exists(os.path.join(OUTPUT_DIR, name)):
            shutil.copy(src, os.path.join(OUTPUT_DIR, name))
    for src in glob.glob(os.path.join(RESUME_DIR, "eval_*.json")):
        if not os.path.exists(os.path.join(OUTPUT_DIR, os.path.basename(src))):
            shutil.copy(src, OUTPUT_DIR)

ck, resume_path = None, None
if RESUME_DIR:
    for name in (LATEST_CKPT, PREV_CKPT):
        p = os.path.join(RESUME_DIR, name)
        if not os.path.exists(p):
            continue
        try:
            ck, resume_path = torch.load(p, map_location=device, weights_only=False), p
            break
        except Exception as e:
            print(f"  !! could not load {p} ({e!r}); trying the fallback checkpoint")
    if ck is None:
        raise RuntimeError(f"no loadable Arch 3 checkpoint in {RESUME_DIR}")

if ck is not None:
    print(f"Resuming from {resume_path}")
    assert ck["model_config"] == asdict(cfg), "architecture config differs from the checkpoint"
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    run = ck["run"]
    for k, v in new_run_state().items():     # fields added after the checkpoint was written
        run.setdefault(k, v)
    block_sampler.load(ck["samplers"]["block"])
    task_sampler.load(ck["samplers"]["task"])
    random.setstate(ck["rng"]["python"]); np.random.set_state(ck["rng"]["numpy"])
    torch.set_rng_state(ck["rng"]["torch"].cpu())            # RNG states must be CPU ByteTensors; map_location moved them
    if torch.cuda.is_available() and ck["rng"]["cuda"] is not None:
        torch.cuda.set_rng_state_all([s.cpu() for s in ck["rng"]["cuda"]])
    del ck
    print(f"  update {run['update']:,} / planned {run['planned_total']} | active training so far "
          f"{run['active_train_sec']/3600:.2f}h | eval {run['eval_sec']/3600:.2f}h | skipped fp16 updates {run['skipped_updates']}")
else:
    print("Starting a new run.")

run["sessions"].append({"start_unix": time.time(), "start_update": run["update"],
                        "data_ready_after_sec": session_elapsed()})
session_deadline = (SESSION_LIMIT_HOURS - SESSION_RESERVE_HOURS) * 3600
last_ckpt = time.time()
win = {o: [0.0, 0] for o in OBJS}   # running (loss sum, tokens) per objective between log lines
consecutive_skips = 0
stop_reason = None
stage = "idle"                      # "step" while parameters may be mid-update (no emergency save then)
model.train()

try:
    while True:
        stage = "idle"
        u = run["update"]
        if run["planned_total"] is not None and u >= run["planned_total"]:
            run["done"] = True
            stop_reason = "planned update horizon reached"
            break
        if session_elapsed() >= session_deadline:
            stop_reason = f"session safety cutoff ({SESSION_LIMIT_HOURS - SESSION_RESERVE_HOURS:.2f}h of session time)"
            break

        # milestones happen at update boundaries, before the update that crosses them
        if run["final_start"] is not None and u >= run["final_start"] and not run["pre_final_saved"]:
            save_checkpoint(FOUNDATION_CKPT, "mixed-objective foundation, before task-matched segment")
            run["pre_final_saved"] = True
            do_eval("prefinal")
            save_checkpoint(LATEST_CKPT, "latest (before task-matched segment)")
            last_ckpt = time.time()
        if run["planned_total"] is not None and run["final_start"] is not None:
            for frac in EVAL_MID_FRACS:
                if frac not in run["mid_evals_done"] and int(frac * run["planned_total"]) <= u < run["final_start"]:
                    run["mid_evals_done"].append(frac)
                    do_eval(f"mid{frac:.2f}")

        t0 = time.time()                    # timed from batch construction, so the horizon reflects real wall time
        stage = "batch"
        obj = objective_for(u, run["final_start"])
        micro = [make_microbatch(obj, u, mi) for mi in range(GRAD_ACCUM)]
        n_valid = sum(int((mb[3] != IGNORE_INDEX).sum()) for mb in micro)
        n_enc = sum(int(mb[1].sum()) for mb in micro)
        lr = lr_at(u, run["planned_total"])
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        stage = "forward"
        optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0
        for mb in micro:
            enc, enc_mask, dec_in, tgt = to_device(mb)
            with amp_ctx():
                logits = model(enc, dec_in, enc_mask)
            loss_tok = F.cross_entropy(logits.float().view(-1, logits.size(-1)), tgt.view(-1),
                                       ignore_index=IGNORE_INDEX, reduction="sum")
            scaler.scale(loss_tok / n_valid).backward()     # mean over all valid target tokens of the whole update
            loss_sum += loss_tok.item()
            del logits, loss_tok
        scaler.unscale_(optimizer)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP))
        scale_before = scaler.get_scale()
        stage = "step"
        scaler.step(optimizer)
        scaler.update()
        if use_amp:
            torch.cuda.synchronize()
        dt = time.time() - t0
        run["active_train_sec"] += dt
        run["attempts"] += 1

        if (use_amp and scaler.get_scale() < scale_before) or not math.isfinite(loss_sum):
            run["skipped_updates"] += 1          # fp16 overflow: no parameter change, LR schedule does not advance
            consecutive_skips += 1
            stage = "idle"
            if consecutive_skips >= 50:
                raise RuntimeError(f"{consecutive_skips} consecutive skipped updates at update {u}; training is unstable")
            continue
        consecutive_skips = 0
        run["update"] = u + 1
        stage = "idle"
        run["updates"][obj] += 1
        run["examples"][obj] += BATCH_SIZE * GRAD_ACCUM
        run["enc_tokens"][obj] += n_enc
        run["dec_loss_tokens"][obj] += n_valid
        win[obj][0] += loss_sum; win[obj][1] += n_valid

        if MEASURE_FROM <= u < MEASURE_TO and obj in run["tp"]:
            run["tp"][obj].append(dt)
        if run["planned_total"] is None and run["update"] >= MEASURE_TO:
            total, fs, info = plan_horizon(run["tp"])
            assert total > 2 * WARMUP_STEPS, f"horizon {total} too short; check throughput"
            run.update(planned_total=total, final_start=fs, horizon_info=info)
            per = info["s_per_update_blended"]
            print(f"\n  >> HORIZON FIXED: {total:,} updates (task-matched segment from update {fs:,}). "
                  f"Measured {info['s_per_update_cont']:.2f}s/update continuation, {info['s_per_update_denoise']:.2f}s denoising, "
                  f"{per:.2f}s blended -> {total*per/3600:.1f}h active training. If real throughput is 15% slower/faster "
                  f"the same horizon takes {total*per*1.15/3600:.1f}h / {total*per*0.85/3600:.1f}h.\n")

        if run["update"] % LOG_EVERY == 0:
            parts = " ".join(f"{o}={win[o][0]/win[o][1]:.3f}" for o in OBJS if win[o][1] > 0)
            print(f"upd {run['update']:>6,}/{run['planned_total'] or '?'} | {parts} | lr {lr:.2e} | gnorm {gnorm:.2f} | "
                  f"scale {scaler.get_scale():.0f} | skipped {run['skipped_updates']} | {dt:.2f}s/upd | "
                  f"train {run['active_train_sec']/3600:.2f}h eval {run['eval_sec']/3600:.2f}h session {session_elapsed()/3600:.2f}h")
            run["history"].append({"update": run["update"], "lr": lr, "gnorm": gnorm, "scale": scaler.get_scale(),
                                   "active_train_sec": run["active_train_sec"],
                                   **{f"loss_{o}": win[o][0] / win[o][1] for o in OBJS if win[o][1] > 0}})
            win = {o: [0.0, 0] for o in OBJS}

        if run["update"] % VAL_EVERY_UPDATES == 0:
            tv = time.time()
            vl = val_loss_cont(MAX_VAL_BATCHES)
            run["eval_sec"] += time.time() - tv
            run["val_history"].append({"update": run["update"], "val_cont_loss": vl})
            print(f"  >> val continuation loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.1f})")

        if run["update"] % CKPT_EVERY_UPDATES == 0 or time.time() - last_ckpt >= CKPT_EVERY_MIN * 60:
            save_checkpoint(LATEST_CKPT, "latest")
            last_ckpt = time.time()

except BaseException as e:
    print(f"\n!! training interrupted by {type(e).__name__}: {e}")
    if stage != "step":     # parameters, optimizer and run state agree, so this is a valid resume point
        try:
            save_checkpoint(LATEST_CKPT, f"emergency save after {type(e).__name__}")
        except BaseException as e2:
            print(f"!! emergency save failed: {e2!r}")
    else:
        print("!! interrupted inside the optimizer step; keeping the last regular checkpoint")
    raise

print(f"\nStopped: {stop_reason}")
run["sessions"][-1].update({"end_unix": time.time(), "end_update": run["update"], "session_elapsed_sec": session_elapsed()})
save_checkpoint(LATEST_CKPT, "latest (end of session)")
TRAINING_COMPLETE = run["done"]

if TRAINING_COMPLETE:
    atomic_save(model.state_dict(), os.path.join(OUTPUT_DIR, "model.pt"))   # before any generation
    print("model.pt saved.")
    do_eval("final")
    final_val_loss = val_loss_cont()
    final_ppl = math.exp(min(final_val_loss, 20))
    run["final_val_cont_loss"] = final_val_loss
    save_checkpoint(LATEST_CKPT, "latest (training complete)")
    print(f"Final val continuation loss {final_val_loss:.4f} (ppl {final_ppl:.2f}) | best Phase 1 checkpoint: {run['best_phase1']}")
else:
    print(f"Training not finished: update {run['update']:,} of {run['planned_total']}. To resume: in a new version, "
          f"add this version's output as an input, set RUN_MODE = \"resume\", and Save & Run All.")
print(f"Active training {run['active_train_sec']/3600:.2f}h | eval {run['eval_sec']/3600:.2f}h | "
      f"updates per objective {run['updates']} | skipped fp16 updates {run['skipped_updates']} | "
      f"eval failures {len(run['eval_failures'])}")


## 16. Training Curves and Evaluation Summary

In [ ]:
import matplotlib.pyplot as plt

h = run["history"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for o, c in (("cont", "tab:blue"), ("denoise", "tab:orange"), ("task", "tab:green")):
    pts = [(x["update"], x[f"loss_{o}"]) for x in h if f"loss_{o}" in x]
    if pts:
        axes[0].plot(*zip(*pts), label=o, color=c)
if run["final_start"]:
    axes[0].axvline(run["final_start"], color="gray", ls="--", label="task-matched segment")
axes[0].set_title("Training loss per objective"); axes[0].set_xlabel("optimizer update"); axes[0].legend(); axes[0].grid(alpha=0.3)
if run["val_history"]:
    axes[1].plot([v["update"] for v in run["val_history"]], [v["val_cont_loss"] for v in run["val_history"]], marker="o", color="tab:red")
axes[1].set_title("Validation continuation loss"); axes[1].set_xlabel("optimizer update"); axes[1].grid(alpha=0.3)
axes[2].plot([x["update"] for x in h], [x["lr"] for x in h])
axes[2].set_title("Learning rate"); axes[2].set_xlabel("optimizer update"); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()
for e in run["evals"]:
    print(f"eval {e['tag']:>10s} @ {e['update']:>6,}: F1 {e['bertscore_F1']:.4f} | val cont loss {e['val_cont_loss']:.4f} | "
          f"rep4 {e['repetition_4gram']:.3f} | short {e['short_outputs']} | cap hits {e['token_cap_hits']}")


## 17. Load Test Data

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Number of test examples: {len(test_data)}")

# show a sample prompt
sample_id = list(test_data.keys())[0]
print(f"\nSample prompt (id={sample_id}):")
print(test_data[sample_id]["text"][:300] + "...")

## 18. Generate Test Predictions (only when training is complete)

In [ ]:
predictions = {}
gen_report = Counter()
gen_errors = []
if not TRAINING_COMPLETE:
    print("Training is not complete; skipping test generation this session.")
else:
    items = list(test_data.items())
    print(f"Generating {len(items)} predictions with the <cont> mode token...")
    t0 = time.time()
    for i, (example_id, record) in enumerate(items):
        try:
            text, info = generate(model, tokenizer, record["text"], seed=SEED + 100000 + i, return_info=True)
        except Exception as e:     # report it; never fabricate filler
            text, info = "", {"n_words": 0, "hit_token_cap": False, "prompt_truncated": False}
            gen_errors.append({"id": example_id, "error": repr(e)})
        predictions[str(example_id)] = {"text": text}
        gen_report["short_outputs"] += info["n_words"] < TARGET_WORDS
        gen_report["token_cap_hits"] += info["hit_token_cap"]
        gen_report["prompt_truncated"] += info["prompt_truncated"]
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(items)} | {time.time()-t0:.0f}s")
    print(f"\nGeneration complete in {(time.time()-t0)/60:.1f} min | {dict(gen_report)} | errors: {len(gen_errors)}")
    for e in gen_errors[:10]:
        print("  ", e)


## 19. Sample Predictions

In [ ]:
for i, (eid, pred) in enumerate(list(predictions.items())[:5]):
    prompt = test_data[eid]["text"]
    print(f"\n{'='*60}")
    print(f"Example {eid}")
    print(f"PROMPT: {prompt[:200]}...")
    print(f"GENERATED ({len(pred['text'].split())} words): {pred['text'][:300]}...")


## 20. Save Predictions & Create Submission

In [ ]:
if TRAINING_COMPLETE:
    PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
    SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(PREDICTION_PATH, arcname="prediction.json")
    print(f"submission.zip created: {SUBMISSION_ZIP} | predictions: {len(predictions)} | "
          f"avg words {sum(len(p['text'].split()) for p in predictions.values())/len(predictions):.1f}")
else:
    print("Training is not complete; no submission this session.")


## 21. Save Metrics

In [ ]:
metrics = {
    "architecture": "encoder_decoder_rope_rmsnorm_swiglu_mixed_objective",
    "params": n_params, "model_config": asdict(cfg), "vocab": {"base": BASE_VOCAB, "total": VOCAB_SIZE},
    "objectives": {"P_DENOISE": P_DENOISE, "FINAL_TASK_FRAC": FINAL_TASK_FRAC,
                   "span_noise_density": SPAN_NOISE_DENSITY, "span_mean_len": SPAN_MEAN_LEN},
    "training_complete": TRAINING_COMPLETE,
    "updates": run["update"], "planned_total": run["planned_total"], "final_start": run["final_start"],
    "horizon_info": run["horizon_info"], "skipped_updates": run["skipped_updates"],
    "updates_per_objective": run["updates"], "examples_per_objective": run["examples"],
    "encoder_tokens": run["enc_tokens"], "decoder_loss_tokens": run["dec_loss_tokens"],
    "active_train_hours": run["active_train_sec"] / 3600, "eval_hours": run["eval_sec"] / 3600,
    "sessions": run["sessions"], "evals": run["evals"], "best_phase1": run["best_phase1"],
    "val_history": run["val_history"], "final_val_cont_loss": run.get("final_val_cont_loss"),
    "generation_config": {"temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
                          "repetition_penalty": REPETITION_PENALTY, "mode_token": "<cont>"},
    "test_generation_report": dict(gen_report), "test_generation_errors": gen_errors,
}
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2, default=float)
print(f"metrics.json saved | training complete: {TRAINING_COMPLETE} | update {run['update']:,}/{run['planned_total']}")
print(f"Files in {OUTPUT_DIR}: {sorted(os.listdir(OUTPUT_DIR))}")
